Damped Newton method for chi=32 truncating to 15,15

Start Newton method after 4 RG steps.

Damping with newton_step=0.5 is activated a couple of times


In [1]:
using Pkg
Pkg.activate(".")
include("Tools.jl")
include("KrylovTechnical.jl")
include("GaugeFixing.jl");
include("./Lab/newton-step-SR.jl");

  Activating project at `~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R`
GiltTNR/GiltTNR2D_essentials.py:113: SyntaxWarning: invalid escape sequence '\ '
  """
GiltTNR/GiltTNR2D_essentials.py:113: SyntaxWarning: invalid escape sequence '\ '
  """


In [2]:
gilt_eps = 6e-6
chi = 32
trunc_shape = [15 15; 15 15; 15 15; 15 15]  # shape to truncate to, not to deal with Gilt tensor dimension oscillations
cg_eps = 1e-10
newton_eps = 1e-9
gilt_pars = Dict(
	"gilt_eps" => gilt_eps,
	"cg_chis" => collect(1:chi),
	"cg_eps" => cg_eps,
	"verbosity" => 1,
	"rotate" => true
)
Jratio = 1.0


relT=1.0
rg_steps = 15
#do rg_steps steps from the critical tensor
initialA_pars = Dict("relT" => relT, "Jratio" => Jratio)
traj = trajectory(initialA_pars, rg_steps, gilt_pars)["A"];
#NB traj consists of PyObjects


traj = traj .|> x -> fix_continuous_gauge(x)[1]; #this is still PyObjects
traj[rg_steps+1], accepted_elements, _ = fix_discrete_gauge(traj[rg_steps+1]; tol = 1e-7);

function fix_discrete_by_accepted_elements_if_possible(x)
	res = x
	try
		res = fix_discrete_gauge(x, accepted_elements)[1]
	catch
		res = fix_discrete_gauge(x)[1]
	end
	return res
end

traj = traj .|> x -> fix_discrete_by_accepted_elements_if_possible(x);
traj = py_to_ju.(traj);
traj = traj .|> x -> x / norm(x); 

for i in 1:length(traj)
    println(i," ",traj[i].shape, traj[i].qhape )
end

distances = Float64[]
for i in 4:length(traj)-1
	if i <= 10
		push!(distances, embedded_distance_with_additional_sign_fixing(traj[i], traj[i+1]))
	else
		try
			push!(distances, embedded_distance(traj[i], traj[i+1]))
		catch
			push!(distances, NaN)
		end
	end
end

println(distances)

A = Any[ NaN for _ in 1:40 ]; # list of tensors, Newton method trajectory
accepted_elements = Any[ NaN for _ in 1:40 ]; # list of elements in gauge-fixing
deltaA = Any[ NaN for _ in 1:40 ]; # list of deltaA's proposed by Newton method

┌ Warning: new_list_of_elements: new entry is below the threshold. It was -9.910499524110562e-5 and became -4.607290536820544e-8. Index CartesianIndex(1, 5, 29, 21) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466
┌ Warning: new_list_of_elements: new entry is below the threshold. It was -9.189427058343429e-5 and became -2.030694655389138e-8. Index CartesianIndex(21, 29, 5, 1) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466
┌ Warning: new_list_of_elements: new entry is below the threshold. It was 2.1827905363925522e-5 and became -3.8691649759337466e-8. Index CartesianIndex(1, 7, 15, 2) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466
┌ Warning: new_list_of_elements: new entry is below the threshold. It was -1.3819853994278374e-5 and became 0.0. Index CartesianIndex(17, 23, 16, 5) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466
┌ Warning: new_list_of_elements:

1 

┌ Warning: new_list_of_elements: new entry is below the threshold. It was 0.00011736532455638104 and became 6.464091962340553e-8. Index CartesianIndex(1, 21, 13, 21) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466
┌ Warning: new_list_of_elements: new entry is below the threshold. It was 1.2642452878832279e-5 and became -9.731593715425455e-8. Index CartesianIndex(21, 32, 7, 1) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466


[1 1; 1 1; 1 1; 1 1][0 1; 0 1; 0 1; 0 1]
2 [2 2; 2 2; 2 2; 2 2][0 1; 0 1; 0 1; 0 1]
3 [8 8; 8 8; 8 8; 8 8][0 1; 0 1; 0 1; 0 1]
4 [16 16; 16 16; 16 16; 16 16][0 1; 0 1; 0 1; 0 1]
5 [16 16; 16 16; 16 16; 16 16][0 1; 0 1; 0 1; 0 1]
6 [15 17; 15 17; 15 17; 15 17][0 1; 0 1; 0 1; 0 1]
7 [16 16; 15 17; 16 16; 15 17][0 1; 0 1; 0 1; 0 1]
8 [16 16; 16 16; 16 16; 16 16][0 1; 0 1; 0 1; 0 1]
9 [16 16; 16 16; 16 16; 16 16][0 1; 0 1; 0 1; 0 1]
10 [16 16; 16 16; 16 16; 16 16][0 1; 0 1; 0 1; 0 1]
11 [16 16; 16 16; 16 16; 16 16][0 1; 0 1; 0 1; 0 1]
12 [16 16; 16 16; 16 16; 16 16][0 1; 0 1; 0 1; 0 1]
13 [16 16; 16 16; 16 16; 16 16][0 1; 0 1; 0 1; 0 1]
14 [16 16; 16 16; 16 16; 16 16][0 1; 0 1; 0 1; 0 1]
15 [16 16; 16 16; 16 16; 16 16][0 1; 0 1; 0 1; 0 1]
16 [16 16; 16 16; 16 16; 16 16][0 1; 0 1; 0 1; 0 1]


┌ Warning: new_list_of_elements: new entry is below the threshold. It was -0.0002891364410801797 and became -1.3056167353890918e-8. Index CartesianIndex(17, 1, 31, 1) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466
┌ Warning: new_list_of_elements: new entry is below the threshold. It was 0.00020902628094827876 and became -3.954377922069119e-8. Index CartesianIndex(1, 31, 1, 17) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466
┌ Warning: construct_linear_system: unable to find 63 independent rows, will fix only 61 dofs
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:338
┌ Warning: new_list_of_elements: new entry is below the threshold. It was -0.00012205254333979844 and became 0.0. Index CartesianIndex(1, 1, 33, 17) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466
┌ Warning: new_list_of_elements: new entry is below the threshold. It was 7.940159563687684e-5 and be

[0.12815309978463302, 0.06302768962029971, 0.06457901452425767, 0.05669906460138519, 0.1148033422286149, 0.11656540662674589, 0.09061758192743109, 0.044627553534162305, 0.1159776667652129, 0.11683735600448537, 0.3729258011794033, 0.38334396871843696]


In [3]:
A[1] = truncate_blocks(traj[8], trunc_shape)
Afin = A[1]
deltaAfin = A[1]
i=0
for i in 1:30
    println("i=",i)
    println("gilt_eps = ", gilt_pars["gilt_eps"])
    A[i], accepted_elements[i] = fix_discrete_gauge(A[i]; tol = 1e-7);
    e0, RAshape = fp_error_with_shape(A[i],accepted_elements[i], gilt_pars; trunc_shape = trunc_shape);
    println("||R(A[i])-A[i]||= ", e0)
    println("shapes:", A[i].shape, RAshape)
    flush(stdout)
    if A[i].shape != RAshape
        throw(ErrorException("shapes unequal"))
    end
    deltaA[i] = newton_correction_with_iterations_fixed(A[i], 10, accepted_elements[i], gilt_pars; trunc_shape = trunc_shape);
    println("||deltaA[i]||= ", norm(deltaA[i]))
    newton_step = 1.0
    enew = e0
    while true #damped Newton method implementation, which reduces a step by 2 if cost function does not decrease
        println("newton_step= ", newton_step)
        Anew = A[i] + newton_step * deltaA[i]
        Anew, accepted_elements_new = fix_discrete_gauge(Anew; tol = 1e-7);
        enew, RAnewshape = fp_error_with_shape(Anew, accepted_elements_new, gilt_pars; trunc_shape = trunc_shape)
        println("fp_error= ", enew)
        println("shapes:", Anew.shape, RAnewshape)
        if enew < e0 && Anew.shape == RAnewshape
            A[i+1] = Anew
            break
        end
        newton_step *= 0.5 
    end
    if enew < newton_eps
        break
    end
end

i=1
gilt_eps = 6.0e-6
||R(A[i])-A[i]||= 0.12405384213095715
shapes:[15 15; 15 15; 15 15; 15 15][15 15; 15 15; 15 15; 15 15]
Dict{Any, Any}((1, "N") => 27, (1, "W") => 22, (1, "S") => 45, (1, "E") => 26, (2, "N") => 1, (2, "W") => 1, (2, "S") => 1, (2, "E") => 1)


┌ Info: Arnoldi eigsolve finished after 3 iterations:
│ *  10 eigenvalues converged
│ *  norm of residuals = (1.5868901953934073e-44, 4.2237348545830204e-34, 2.2272795658795225e-26, 1.5131220294657013e-19, 1.4639042550110635e-18, 7.57298247521365e-16, 7.57298247521365e-16, 2.2032973131810598e-14, 7.587090310403987e-15, 7.587090310403987e-15)
└ *  number of operations = 51


EIGENVALUES (INITIAL):
-8.07737782493284 + 0.0im
1.9957230699163355 + 0.0im
1.253560853578314 + 0.0im
-0.9479546774106435 + 0.0im
-0.9053220779782254 + 0.0im
0.6583890024341963 + 0.2909244813432157im
0.6583890024341963 - 0.2909244813432157im
0.6598752689503701 + 0.0im
-0.23109279960511375 + 0.5967428440423735im
-0.23109279960511375 - 0.5967428440423735im
||deltaA[i]||= 0.3299511193932376
newton_step= 1.0
fp_error= 0.27639642184218594
shapes:[15 15; 15 15; 15 15; 15 15][15 15; 15 15; 15 15; 15 15]
newton_step= 0.5
fp_error= 0.17478062000006114
shapes:[15 15; 15 15; 15 15; 15 15][15 15; 15 15; 15 15; 15 15]
newton_step= 0.25
fp_error= 0.10625943809016003
shapes:[15 15; 15 15; 15 15; 15 15][15 15; 15 15; 15 15; 15 15]
i=2
gilt_eps = 6.0e-6
||R(A[i])-A[i]||= 0.10625943809016003
shapes:[15 15; 15 15; 15 15; 15 15][15 15; 15 15; 15 15; 15 15]
Dict{Any, Any}((1, "N") => 34, (1, "W") => 30, (1, "S") => 39, (1, "E") => 39, (2, "N") => 1, (2, "W") => 1, (2, "S") => 1, (2, "E") => 1)


LoadError: InterruptException:

In [3]:
A[1] = truncate_blocks(traj[16], trunc_shape);

In [4]:
display(gilt_pars)
i=1;
A[i], accepted_elements[i] = fix_discrete_gauge(A[i]; tol = 1e-7);
deltaA[i] = newton_correction_with_iterations_fixed(A[i], 10, accepted_elements[i], gilt_pars; trunc_shape = trunc_shape);

Dict{String, Any} with 5 entries:
  "gilt_eps"  => 6.0e-6
  "rotate"    => true
  "cg_eps"    => 1.0e-10
  "verbosity" => 2
  "cg_chis"   => [1, 2, 3, 4, 5, 6, 7, 8, 9, 10  …  23, 24, 25, 26, 27, 28, 29,…

2024-12-30 18:33:09 I: Lap                                 1
2024-12-30 18:33:09 I: Gilt initiated at leg               S
depth_dictionary: {(1, 'S'): 7}
2024-12-30 18:33:09 I: Leg status:                         False
2024-12-30 18:33:09 I: Gilt info,                          Error = 2.590e-05, shape = (np.int64(30), np.int64(30), 10, np.int64(30)) [(np.int64(30), np.int64(30), np.int64(30), np.int64(30))]
2024-12-30 18:33:09 I: Gilt initiated at leg               N
depth_dictionary: {(1, 'S'): 7, (1, 'N'): 14}
2024-12-30 18:33:09 I: Leg status:                         False
2024-12-30 18:33:09 I: Gilt info,                          Error = 5.200e-05, shape = (10, np.int64(30), 10, np.int64(30)) [(np.int64(30), np.int64(30), np.int64(30), np.int64(30))]
2024-12-30 18:33:09 I: Gilt initiated at leg               E
depth_dictionary: {(1, 'S'): 7, (1, 'N'): 14, (1, 'E'): 13}
2024-12-30 18:33:10 I: Leg status:                         False
2024-12-30 18:33:10 I: Gilt info,                

--- Logging error ---
Traceback (most recent call last):
  File "/Users/slava/.julia/conda/3/aarch64/lib/python3.12/logging/__init__.py", line 1164, in emit
    self.flush()
  File "/Users/slava/.julia/conda/3/aarch64/lib/python3.12/logging/__init__.py", line 1144, in flush
    self.stream.flush()
BlockingIOError: [Errno 35] write could not complete without blocking
Call stack:
  File "GiltTNR/GiltTNR2D_essentials.py", line 95, in gilttnr_step
    A, log_fact, err_A_split1, err_A_split2, SB2, SC2 = trg(A, A, log_fact, pars)
  File "GiltTNR/GiltTNR2D_essentials.py", line 152, in trg
    status_print("TRG splitting,")
  File "GiltTNR/GiltTNR2D_essentials.py", line 217, in status_print
    logging.info(status_str)
Message: 'TRG splitting,                      '
Arguments: ()
--- Logging error ---
Traceback (most recent call last):
  File "/Users/slava/.julia/conda/3/aarch64/lib/python3.12/logging/__init__.py", line 1164, in emit
    self.flush()
  File "/Users/slava/.julia/conda/3/aarch64

EIGENVALUES (INITIAL):
2.6764445446358027 + 0.0im
0.32974932147463354 + 1.1656391071662118im
0.32974932147463354 - 1.1656391071662118im
-0.9848480261868375 + 0.6628389651509059im
-0.9848480261868375 - 0.6628389651509059im
0.9953016999806322 + 0.37481665089408917im
0.9953016999806322 - 0.37481665089408917im
-0.07100100963525245 + 0.7850631660707236im
-0.07100100963525245 - 0.7850631660707236im
0.5468440593352452 + 0.409555174079101im
0.5468440593352452 - 0.409555174079101im
 
2024-12-30 18:33:31 I: TRG splitting, done.                Error = 6.999e-04, chi = 32
2024-12-30 18:33:31 I: TRG splitting,                      
2024-12-30 18:33:32 I: TRG splitting, done.                Error = 4.976e-04, chi = 32
2024-12-30 18:33:32 I: TRG contracting,                    
2024-12-30 18:33:32 I: TRG contracting, done.              
2024-12-30 18:33:32 I: Bond repetitions is set to:         2, convergence signals will be ignored
2024-12-30 18:33:32 I: Lap                                 1
2024-12

 = trg(A, A, log_fact, pars)
--- Logging error ---
--- Logging error ---
--- Logging error ---
--- Logging error ---
--- Logging error ---
--- Logging error ---
--- Logging error ---
--- Logging error ---
--- Logging error ---
--- Logging error ---
--- Logging error ---
--- Logging error ---
--- Logging error ---
--- Logging error ---
--- Logging error ---
--- Logging error ---
--- Logging error ---
--- Logging error ---
--- Logging error ---
--- Logging error ---
--- Logging error ---
--- Logging error ---
--- Logging error ---
--- Logging error ---
--- Logging error ---
--- Logging error ---
--- Logging error ---
--- Logging error ---
--- Logging error ---
--- Logging error ---
--- Logging error ---
--- Logging error ---
--- Logging error ---
--- Logging error ---
--- Logging error ---
--- Logging error ---
--- Logging error ---
--- Logging error ---
--- Logging error ---
--- Logging error ---
--- Logging error ---
--- Logging error ---
--- Logging error ---
--- Logging error ---
---